In [ ]:
from langgraph.graph import StateGraph , START , END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
model = ChatOpenAI()

In [2]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [ ]:
def create_outline(state: BlogState) -> BlogState:
    #fetch title 
    title = state['title']
    #create prompt
    prompt = f"Create a detailed outline for a blog post titled: {title}"
    
    outline = model.invoke(prompt).content
    state['outline'] = outline
    return state

In [ ]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f"Write a detailed blog post based on the following outline:\n{outline}"
    
    content = model.invoke(prompt).content
    state['content'] = content
    return state

In [ ]:
graph = StateGraph(BlogState)

#create nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

#add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)


workflow = graph.compile()

In [ ]:
initial_state = {'title': 'The Future of Artificial Intelligence'}
final_state = workflow.invoke(initial_state)
print(final_state)